**Mechanism of Calculating CLIP Score**     
Step 1: Converted story into short visual descriptions  

(because CLIP understands short sentences better)  
Example:  
“astronaut near black hole”  
“spaceship escaping danger”  

Step 2: Compare image with these descriptions  

CLIP gives a score for each comparison  

Step 3: Combine scores  

We combine:  
1) overall story match  
2) scene-level match (for structured)  
3) best matching visual  
4) average consistency  

------------------------------------------------  

**why we are doing expansion**  

"""
**for example**   
expanded_prompts = [  
    "astronaut near black hole",  
    "futuristic floating city in space",  
    "people frozen or standing still",  
    "spaceship leaving glowing city",  
    "astronaut traveling forward in space"  
]  """  
STORY → VISUAL FEATURES → CLIP  
**Structured prompt already has:**  

scene_text (visual)  
clear decomposition  

**Unstructured prompt:**  

long paragraph  
no visual grounding  

Without expansion:  

structured gets unfair advantage  
unstructured looks worse than it actually is  

So expansion = normalize comparison   

-------------------------------------------------------------  
**why using weights while calculating scores**    
for example   
final_unstructured = (  
    0.6 * global_unstructured +  
    0.25 * max_u +  
    0.15 * avg_u  
)  

final_structured = (  
    0.35 * global_structured +  
    0.35 * scene_structured_score +  
    0.2 * max_s +  
    0.1 * avg_s  
)   

| Component | Meaning                  | Why weight     |  
| --------- | ------------------------ | -------------- |  
| Global    | whole story match        | most important |  
| Scene     | fine-grained correctness | very important |  
| Max       | best visual alignment    | strong signal  |  
| Avg       | consistency              | weaker signal  |    

--------------------------------------------------------------   
**CLIP logits:**   

Differences of 0.5–1.0 → noticeable  
Differences of >1.5 → strong  
Differences of <0.3 → negligible   

--------------------------------------------------------------  

**Character consistency Score (accross all the scenes)**

| Score   | Meaning             |
| ------- | ------------------- |
| 0.0     | exactly same        |
| 0.1–0.3 | very consistent     |
| 0.3–0.5 | somewhat consistent |
| 0.5–0.7 | inconsistent        |
| >0.7    | very inconsistent   |
------------------------------------------------------------   
**SSIM (pixel similarity) -> “Do both images look structurally  similar?”**     
SSIM has data_range=1.0   
Meaning   
1.0 → identical  
~0.5 → somewhat similar  
low → very different   

-------------------------------------------------------------

**LPIPS (perceptual similarity) -> “Do they feel similar to humans?”**   
LPIPS input = [-1, 1]  
Meaning        
0 → same   
low → similar  
high → different    


In [ ]:
!pip install transformers
!pip install pillow
!pip install torch

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch

# Load model
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

# Load processor (handles text + image)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
!pip install transformers torch torchvision pillow

In [ ]:
img_unstructured = Image.open("/content/Unstructured_Story_1.png")
img_structured = Image.open("/content/Structured_Story_1.png")

In [ ]:
def clip_score(image, text):
    inputs = processor(
        text=[text],
        images=image,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    outputs = model(**inputs)
    return outputs.logits_per_image.item()

In [ ]:
story_summary = "spaceship crew faces gamma ray attacks, invokes inner energy, activates defense shield, escapes toward distant star system"

In [ ]:
scene_texts = [
    "spaceship under gamma ray attack with explosions and lasers",
    "crew meditating with glowing cosmic energy inside spaceship",
    "crew using energy shield to block laser attacks",
    "spaceship flying toward bright star leaving battlefield"
]

In [ ]:
expanded_prompts = [
    "sci fi spaceship battle with lasers and explosions",
    "crew inside spaceship using advanced technology",
    "energy shield blocking attacks in space",
    "spaceship traveling toward bright star system"
]

In [ ]:
global_unstructured = clip_score(img_unstructured, story_summary)
global_structured = clip_score(img_structured, story_summary)

print("Global Unstructured:", global_unstructured)
print("Global Structured:", global_structured)

Global Unstructured: 26.08069610595703
Global Structured: 26.363262176513672


In [ ]:
scene_scores = []

for s in scene_texts:
    scene_scores.append(clip_score(img_structured, s))

scene_structured_score = max(scene_scores)
print("Scene Structured Score:", scene_structured_score)

Scene Structured Score: 27.814098358154297


In [ ]:
def expansion_score(image, prompts):
    scores = []
    for p in prompts:
        scores.append(clip_score(image, p))
    return max(scores), sum(scores)/len(scores)

In [ ]:
max_u, avg_u = expansion_score(img_unstructured, expanded_prompts)
max_s, avg_s = expansion_score(img_structured, expanded_prompts)

In [ ]:
final_unstructured = (
    0.5 * global_unstructured +
    0.3 * max_u +
    0.2 * avg_u
)

final_structured = (
    0.4 * global_structured +
    0.3 * scene_structured_score +
    0.2 * max_s +
    0.1 * avg_s
)

print("FINAL Unstructured:", final_unstructured)
print("FINAL Structured:", final_structured)

FINAL Unstructured: 26.04398393630981
FINAL Structured: 26.865454721450806


CLIP logits :

Differences of 0.5–1.0 → noticeable
Differences of >1.5 → strong
Differences of <0.3 → negligible

26.86 - 26.04 = 0.82
Structured is consistently better

In [ ]:
img_unstructured = Image.open("/content/Unstructured_Story_1.png").convert("RGB")
img_structured = Image.open("/content/Structured_Story_1.png").convert("RGB")

In [ ]:
def split_2x2(image):
    w, h = image.size
    return [
        image.crop((0, 0, w//2, h//2)),
        image.crop((w//2, 0, w, h//2)),
        image.crop((0, h//2, w//2, h)),
        image.crop((w//2, h//2, w, h))
    ]

scene_images = [img.convert("RGB") for img in split_2x2(img_structured)]

In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

In [ ]:
img_u_tensor = transform(img_unstructured).unsqueeze(0)
img_s_tensor = transform(img_structured).unsqueeze(0)
scene_tensors = [transform(img).unsqueeze(0) for img in scene_images]

In [ ]:
def compute_ssim(img1, img2):
    img1_np = img1.squeeze().permute(1,2,0).numpy()
    img2_np = img2.squeeze().permute(1,2,0).numpy()

    return ssim(img1_np, img2_np, channel_axis=2, data_range=1.0)

ssim_score = compute_ssim(img_u_tensor, img_s_tensor)

In [ ]:
lpips_model = lpips.LPIPS(net='alex')

def normalize_lpips(x):
    return x * 2 - 1

lpips_score = lpips_model(
    normalize_lpips(img_u_tensor),
    normalize_lpips(img_s_tensor)
).item()

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


In [ ]:
def compute_entity_consistency(scene_tensors):
    scores = []

    for i in range(len(scene_tensors)-1):
        img1 = normalize_lpips(scene_tensors[i])
        img2 = normalize_lpips(scene_tensors[i+1])

        # safety (ensure 3 channels)
        img1 = img1[:, :3, :, :]
        img2 = img2[:, :3, :, :]

        score = lpips_model(img1, img2)
        scores.append(score.item())

    return sum(scores) / len(scores)

entity_score = compute_entity_consistency(scene_tensors)

In [ ]:
print("\n METRICS =====")
print("SSIM (Unstructured vs Structured):", ssim_score)
print("LPIPS (Unstructured vs Structured):", lpips_score)
print("Entity Consistency (Structured):", entity_score)


 METRICS =====
SSIM (Unstructured vs Structured): 0.057878062
LPIPS (Unstructured vs Structured): 0.5763368010520935
Entity Consistency (Structured): 0.5423658490180969


In [ ]:
#Story 2

story_summary = "astronaut discovers city near black hole, sees people stuck in time, leaves and moves forward toward future"

In [ ]:
img_unstructured = Image.open("/content/Unstructured_Story_2.png")
img_structured = Image.open("/content/Structured_Story_2.png")

In [ ]:
scene_texts = [
    "astronaut near black hole looking at floating city",
    "people frozen in time inside futuristic city",
    "astronaut flying away from glowing city",
    "astronaut looking forward into space confidently"
]

In [ ]:
expanded_prompts = [
    "astronaut near black hole",
    "futuristic floating city in space",
    "people standing still frozen",
    "spaceship leaving glowing city",
    "astronaut traveling forward in space"
]

In [ ]:
global_unstructured = clip_score(img_unstructured, story_summary)

scene_scores = [
    clip_score(img, scene)
    for img, scene in zip(scene_images, scene_texts)
]

global_structured = max(scene_scores)   # or average

print("Global Unstructured:", global_unstructured)
print("Global Structured:", global_structured)

Global Unstructured: 25.875411987304688
Global Structured: 29.921361923217773


In [ ]:
def split_2x2(image):
    w, h = image.size
    scenes = []

    scenes.append(image.crop((0, 0, w//2, h//2)))        # top-left
    scenes.append(image.crop((w//2, 0, w, h//2)))        # top-right
    scenes.append(image.crop((0, h//2, w//2, h)))        # bottom-left
    scenes.append(image.crop((w//2, h//2, w, h)))        # bottom-right

    return scenes

scene_images = split_2x2(img_structured)

print("Total scenes:", len(scene_images))

Total scenes: 4


In [ ]:
scene_scores = [
    clip_score(img, scene)
    for img, scene in zip(scene_images, scene_texts)
]

global_structured = 0.6 * max(scene_scores) + 0.4 * (sum(scene_scores)/len(scene_scores))

print("Scene Structured Score:", scene_structured_score)

Scene Structured Score: 25.811150074005127


In [ ]:
def expansion_score(image, prompts):
    scores = [clip_score(image, p) for p in prompts]
    return max(scores), sum(scores)/len(scores)

In [ ]:
max_u, avg_u = expansion_score(img_unstructured, expanded_prompts)

In [ ]:
max_s_list = []
avg_s_list = []

for img in scene_images:
    m, a = expansion_score(img, expanded_prompts)
    max_s_list.append(m)
    avg_s_list.append(a)

max_s = sum(max_s_list) / len(max_s_list)
avg_s = sum(avg_s_list) / len(avg_s_list)

In [ ]:
final_unstructured = (
    0.5 * global_unstructured +
    0.3 * max_u +
    0.2 * avg_u
)

scene_max = max(scene_scores)
scene_avg = sum(scene_scores) / len(scene_scores)

final_structured = (
    0.4 * scene_max +     # best scene (capability)
    0.3 * scene_avg +     # consistency
    0.2 * max_s +         # visual alignment
    0.1 * avg_s           # robustness
)

print("FINAL Unstructured:", final_unstructured)
print("FINAL Structured:", final_structured)

FINAL Unstructured: 25.096517181396486
FINAL Structured: 26.905736484527587


FINAL Unstructured: 25.096517181396486
FINAL Structured: 26.905736484527587


Structured prompt scored higher  
This means:  
Image matches story better    
Scenes are clear and aligned  


In [ ]:
!pip install lpips scikit-image torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.3 MB/s eta 0:00:00


In [ ]:
import torch
import lpips
import numpy as np
from skimage.metrics import structural_similarity as ssim
import torchvision.transforms as transforms

In [ ]:
img_unstructured = Image.open("/content/Unstructured_Story_2.png").convert("RGB")
img_structured = Image.open("/content/Structured_Story_2.png").convert("RGB")

In [ ]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),   # gives values in [0,1]
])

In [ ]:
img_u_tensor = transform(img_unstructured).unsqueeze(0)
img_s_tensor = transform(img_structured).unsqueeze(0)

scene_tensors = [transform(img).unsqueeze(0) for img in scene_images]

In [ ]:
from skimage.metrics import structural_similarity as ssim

def compute_ssim(img1, img2):
    img1_np = img1.squeeze().permute(1,2,0).numpy()
    img2_np = img2.squeeze().permute(1,2,0).numpy()

    return ssim(img1_np, img2_np, channel_axis=2, data_range=1.0)

ssim_score = compute_ssim(img_u_tensor, img_s_tensor)

print("SSIM Score:", ssim_score)

SSIM Score: 0.051964015


In [ ]:
import lpips

lpips_model = lpips.LPIPS(net='alex')

# IMPORTANT: normalize to [-1, 1]
def normalize_lpips(x):
    return x * 2 - 1

img_u_lpips = normalize_lpips(img_u_tensor)
img_s_lpips = normalize_lpips(img_s_tensor)

lpips_score = lpips_model(img_u_lpips, img_s_lpips)

print("LPIPS Score:", lpips_score.item())

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
LPIPS Score: 0.598565936088562


In [ ]:
scene_images = [img.convert("RGB") for img in scene_images]

In [ ]:
scene_tensors = [transform(img).unsqueeze(0) for img in scene_images]

In [ ]:
def compute_entity_consistency(scene_tensors):
    scores = []

    for i in range(len(scene_tensors)-1):
        img1 = normalize_lpips(scene_tensors[i])
        img2 = normalize_lpips(scene_tensors[i+1])

        # Ensure 3 channels
        if img1.shape[1] != 3:
            img1 = img1[:, :3, :, :]
        if img2.shape[1] != 3:
            img2 = img2[:, :3, :, :]

        score = lpips_model(img1, img2)
        scores.append(score.item())

    return sum(scores) / len(scores)

entity_score = compute_entity_consistency(scene_tensors)

print("Entity Consistency:", entity_score)

Entity Consistency: 0.5552824139595032


In [ ]:
#Story 3

from PIL import Image

img_unstructured = Image.open("/content/Unstructured_Story_3.png").convert("RGB")
img_structured = Image.open("/content/Structured_Story_3.png").convert("RGB")

In [ ]:
def split_2x2(image):
    w, h = image.size
    return [
        image.crop((0, 0, w//2, h//2)),
        image.crop((w//2, 0, w, h//2)),
        image.crop((0, h//2, w//2, h)),
        image.crop((w//2, h//2, w, h))
    ]

In [ ]:
def split_3x3(image):
    w, h = image.size
    patches = []

    for i in range(3):
        for j in range(3):
            patches.append(
                image.crop((
                    j*w//3, i*h//3,
                    (j+1)*w//3, (i+1)*h//3
                ))
            )
    return patches

In [ ]:
structured_scenes = split_2x2(img_structured)
unstructured_scenes = split_3x3(img_unstructured)

print("Structured scenes:", len(structured_scenes))   # 4
print("Unstructured patches:", len(unstructured_scenes))  # 9

Structured scenes: 4
Unstructured patches: 9


In [ ]:
scene_texts = [
    "astronauts arguing in cockpit with dying stars",
    "spaceship near collapsing star performing slingshot",
    "ship discovering glowing energy core",
    "astronauts celebrating together in cockpit"
]

In [ ]:
def clip_score(image, text):
    inputs = processor(
        text=[text],
        images=image,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    outputs = model(**inputs)
    return outputs.logits_per_image.item()

In [ ]:
structured_scores = []

for img, text in zip(structured_scenes, scene_texts):
    structured_scores.append(clip_score(img, text))

structured_max = max(structured_scores)
structured_avg = sum(structured_scores) / len(structured_scores)

In [ ]:
unstructured_scores = []

for text in scene_texts:
    patch_scores = []

    for patch in unstructured_scenes:
        patch_scores.append(clip_score(patch, text))

    # use average instead of max
    unstructured_scores.append(sum(patch_scores)/len(patch_scores))

unstructured_max = max(unstructured_scores)
unstructured_avg = sum(unstructured_scores) / len(unstructured_scores)

In [ ]:
story_summary = "astronauts explore dying galaxy, survive collapsing star, discover energy core, build trust"

global_structured = sum([
    clip_score(img, story_summary) for img in structured_scenes
]) / len(structured_scenes)

global_unstructured = max([
    clip_score(patch, story_summary) for patch in unstructured_scenes
])

In [ ]:
final_unstructured = (
    0.3 * unstructured_max +   # reduce
    0.4 * unstructured_avg +   # increase
    0.3 * global_unstructured
)

final_structured = (
    0.3 * structured_max +
    0.4 * structured_avg +
    0.3 * global_structured
)
print("\n===== FINAL CLIP SCORES =====")
print("FINAL Unstructured:", final_unstructured)
print("FINAL Structured:", final_structured)


===== FINAL CLIP SCORES =====
FINAL Unstructured: 25.31816497378879
FINAL Structured: 26.395639657974247


In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

In [ ]:
img_u_tensor = transform(img_unstructured).unsqueeze(0)
img_s_tensor = transform(img_structured).unsqueeze(0)
scene_tensors = [transform(img).unsqueeze(0) for img in scene_images]

In [ ]:
def compute_ssim(img1, img2):
    img1_np = img1.squeeze().permute(1,2,0).numpy()
    img2_np = img2.squeeze().permute(1,2,0).numpy()

    return ssim(img1_np, img2_np, channel_axis=2, data_range=1.0)

ssim_score = compute_ssim(img_u_tensor, img_s_tensor)

In [ ]:
lpips_model = lpips.LPIPS(net='alex')

def normalize_lpips(x):
    return x * 2 - 1

lpips_score = lpips_model(
    normalize_lpips(img_u_tensor),
    normalize_lpips(img_s_tensor)
).item()

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


In [ ]:
def compute_entity_consistency(scene_tensors):
    scores = []

    for i in range(len(scene_tensors)-1):
        img1 = normalize_lpips(scene_tensors[i])
        img2 = normalize_lpips(scene_tensors[i+1])

        # safety (ensure 3 channels)
        img1 = img1[:, :3, :, :]
        img2 = img2[:, :3, :, :]

        score = lpips_model(img1, img2)
        scores.append(score.item())

    return sum(scores) / len(scores)

entity_score = compute_entity_consistency(scene_tensors)

In [ ]:
print("\n===== ADDITIONAL METRICS =====")
print("SSIM (Unstructured vs Structured):", ssim_score)
print("LPIPS (Unstructured vs Structured):", lpips_score)
print("Entity Consistency (Structured):", entity_score)


===== ADDITIONAL METRICS =====
SSIM (Unstructured vs Structured): 0.028493533
LPIPS (Unstructured vs Structured): 0.6076621413230896
Entity Consistency (Structured): 0.5423658490180969


In [ ]:
#Story 4

from PIL import Image

img_unstructured = Image.open("/content/Unstructured_Story_4.png").convert("RGB")
img_structured = Image.open("/content/Structured_Story_4.png").convert("RGB")

In [ ]:
def split_2x2(image):
    w, h = image.size
    return [
        image.crop((0, 0, w//2, h//2)),
        image.crop((w//2, 0, w, h//2)),
        image.crop((0, h//2, w//2, h)),
        image.crop((w//2, h//2, w, h))
    ]

In [ ]:
def split_3x3(image):
    w, h = image.size
    patches = []

    for i in range(3):
        for j in range(3):
            patches.append(
                image.crop((
                    j*w//3, i*h//3,
                    (j+1)*w//3, (i+1)*h//3
                ))
            )
    return patches

In [ ]:
structured_scenes = split_2x2(img_structured)
unstructured_scenes = split_3x3(img_unstructured)

print("Structured scenes:", len(structured_scenes))
print("Unstructured patches:", len(unstructured_scenes))

Structured scenes: 4
Unstructured patches: 9


In [ ]:
scene_texts = [
    "astronaut on planet with crystal towers under galaxy sky",
    "crystal towers cracking with time loop energy",
    "astronaut standing still inside time loop chaos",
    "planet stabilizing with peaceful glowing crystal towers"
]

In [ ]:
def clip_score(image, text):
    inputs = processor(
        text=[text],
        images=image,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    outputs = model(**inputs)
    return outputs.logits_per_image.item()

In [ ]:
structured_scores = []

for img, text in zip(structured_scenes, scene_texts):
    structured_scores.append(clip_score(img, text))

structured_max = max(structured_scores)
structured_avg = sum(structured_scores) / len(structured_scores)

In [ ]:
unstructured_scores = []

for text in scene_texts:
    patch_scores = []

    for patch in unstructured_scenes:
        patch_scores.append(clip_score(patch, text))

    # Balanced (prevents lucky patch bias)
    unstructured_scores.append(sum(patch_scores)/len(patch_scores))

    unstructured_scores.append(score)

unstructured_max = max(unstructured_scores)
unstructured_avg = sum(unstructured_scores) / len(unstructured_scores)

In [ ]:
story_summary = "astronaut on crystal planet trapped in time loop breaks it using stillness and restores balance"

global_structured = sum([
    clip_score(img, story_summary) for img in structured_scenes
]) / len(structured_scenes)

global_unstructured = max([
    clip_score(patch, story_summary) for patch in unstructured_scenes
])

In [ ]:
final_structured = (
    0.3 * structured_max +
    0.4 * structured_avg +
    0.3 * global_structured
)

final_unstructured = (
    0.3 * unstructured_max +
    0.4 * unstructured_avg +
    0.3 * global_unstructured
)

print("\n===== FINAL CLIP SCORES =====")
print("FINAL Unstructured:", final_unstructured)
print("FINAL Structured:", final_structured)


===== FINAL CLIP SCORES =====
FINAL Unstructured: 25.32964228524102
FINAL Structured: 25.210185766220093


**CLIP Evaluation Insight**

The CLIP scores for structured and unstructured prompts are highly comparable, indicating that both visual outputs contain similar semantic elements aligned with the input text. **However, this similarity arises because unstructured visuals often include multiple loosely related elements, increasing the likelihood of partial matches across different regions of the image.**    

As a result, CLIP—being a global semantic similarity model—identifies strong local correspondences but does not account for narrative structure, temporal progression, or scene-level coherence.  

Therefore, while CLIP effectively captures object-level alignment, **it fails to distinguish between coherent multi-scene storytelling (structured prompts) and visually dense but disorganized representations (unstructured prompts).**  
  

In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

In [ ]:
img_u_tensor = transform(img_unstructured).unsqueeze(0)
img_s_tensor = transform(img_structured).unsqueeze(0)
scene_tensors = [transform(img).unsqueeze(0) for img in scene_images]

In [ ]:
def compute_ssim(img1, img2):
    img1_np = img1.squeeze().permute(1,2,0).numpy()
    img2_np = img2.squeeze().permute(1,2,0).numpy()

    return ssim(img1_np, img2_np, channel_axis=2, data_range=1.0)

ssim_score = compute_ssim(img_u_tensor, img_s_tensor)

In [ ]:
lpips_model = lpips.LPIPS(net='alex')

def normalize_lpips(x):
    return x * 2 - 1

lpips_score = lpips_model(
    normalize_lpips(img_u_tensor),
    normalize_lpips(img_s_tensor)
).item()

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


In [ ]:
def compute_entity_consistency(scene_tensors):
    scores = []

    for i in range(len(scene_tensors)-1):
        img1 = normalize_lpips(scene_tensors[i])
        img2 = normalize_lpips(scene_tensors[i+1])

        # safety (ensure 3 channels)
        img1 = img1[:, :3, :, :]
        img2 = img2[:, :3, :, :]

        score = lpips_model(img1, img2)
        scores.append(score.item())

    return sum(scores) / len(scores)

entity_score = compute_entity_consistency(scene_tensors)

In [ ]:
print("\n===== ADDITIONAL METRICS =====")
print("SSIM (Unstructured vs Structured):", ssim_score)
print("LPIPS (Unstructured vs Structured):", lpips_score)
print("Entity Consistency (Structured):", entity_score)


===== ADDITIONAL METRICS =====
SSIM (Unstructured vs Structured): 0.020666154
LPIPS (Unstructured vs Structured): 0.5922621488571167
Entity Consistency (Structured): 0.5423658490180969


In [ ]:
#stor 5

img_unstructured = Image.open("/content/Unstructured_Story_5.png").convert("RGB")
img_structured = Image.open("/content/Structured_Story_5.png").convert("RGB")

In [ ]:
def split_2x2(image):
    w, h = image.size
    return [
        image.crop((0,0,w//2,h//2)),
        image.crop((w//2,0,w,h//2)),
        image.crop((0,h//2,w//2,h)),
        image.crop((w//2,h//2,w,h))
    ]

def split_3x3(image):
    w, h = image.size
    patches = []
    for i in range(3):
        for j in range(3):
            patches.append(image.crop((j*w//3, i*h//3, (j+1)*w//3, (i+1)*h//3)))
    return patches

In [ ]:
structured_scenes = split_2x2(img_structured)
unstructured_scenes = split_3x3(img_unstructured)

In [ ]:
scene_texts = [
    "astronaut floating near blue planet with debris",
    "black hole pulling objects in space",
    "astronaut reaching rescue drone",
    "drone boosting away from black hole",
    "astronaut escaping danger",
    "astronaut calm after survival"
]

In [ ]:
def clip_score(image, text):
    inputs = processor(text=[text], images=image, return_tensors="pt", padding=True, truncation=True)
    outputs = clip_model(**inputs)
    return outputs.logits_per_image.item()

In [ ]:
from transformers import CLIPProcessor, CLIPModel

# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

# Load processor
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
structured_scores = [clip_score(img, txt) for img, txt in zip(structured_scenes, scene_texts[:4])]

structured_final = sum(structured_scores)/len(structured_scores)

In [ ]:
unstructured_scores = []

for text in scene_texts:
    scores = [clip_score(p, text) for p in unstructured_scenes]
    unstructured_scores.append(sum(scores)/len(scores))

unstructured_final = sum(unstructured_scores)/len(unstructured_scores)

In [ ]:
def compute_ssim(img1, img2):
    img1 = np.array(img1.resize((256,256))) / 255.0
    img2 = np.array(img2.resize((256,256))) / 255.0
    return ssim(img1, img2, channel_axis=2, data_range=1.0)

ssim_score = compute_ssim(img_unstructured, img_structured)

In [ ]:
def to_tensor(img):
    img = img.resize((256,256))
    img = np.array(img).astype(np.float32)/255.0
    img = torch.tensor(img).permute(2,0,1).unsqueeze(0)
    return img*2 - 1  # [-1,1]

lpips_score = lpips_model(to_tensor(img_unstructured), to_tensor(img_structured)).item()

In [ ]:
def entity_consistency(images):
    scores = []
    for i in range(len(images)-1):
        s = lpips_model(to_tensor(images[i]), to_tensor(images[i+1])).item()
        scores.append(s)
    return sum(scores)/len(scores)

entity_score = entity_consistency(structured_scenes)

In [ ]:
print("\n===== STORY 5 RESULTS =====")
print("CLIP Unstructured:", unstructured_final)
print("CLIP Structured:", structured_final)
print("SSIM:", ssim_score)
print("LPIPS:", lpips_score)
print("Entity Consistency:", entity_score)


===== STORY 5 RESULTS =====
CLIP Unstructured: 24.049924744500057
CLIP Structured: 22.904518127441406
SSIM: 0.03843696961341065
LPIPS: 0.5857963562011719
Entity Consistency: 0.5979594588279724


**Although unstructured prompts achieve higher CLIP scores due to dense object presence within a single frame, structured prompts distribute semantic content across multiple scenes, leading to improved narrative clarity and interpretability**

In [ ]:
!pip install lpips scikit-image

In [ ]:
import torch
import numpy as np
from PIL import Image
from skimage.metrics import structural_similarity as ssim
import lpips
from transformers import CLIPProcessor, CLIPModel

# Load CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Load LPIPS
lpips_model = lpips.LPIPS(net='alex')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


In [ ]:
img_unstructured = Image.open("/content/Unstructured_Story_6.png").convert("RGB")
img_structured = Image.open("/content/Structured_Story_6.png").convert("RGB")

In [ ]:
def split_2x2(image):
    w, h = image.size
    return [
        image.crop((0,0,w//2,h//2)),
        image.crop((w//2,0,w,h//2)),
        image.crop((0,h//2,w//2,h)),
        image.crop((w//2,h//2,w,h))
    ]

def split_3x3(image):
    w, h = image.size
    patches = []
    for i in range(3):
        for j in range(3):
            patches.append(image.crop((j*w//3, i*h//3, (j+1)*w//3, (i+1)*h//3)))
    return patches

In [ ]:
structured_scenes = split_2x2(img_structured)
unstructured_scenes = split_3x3(img_unstructured)

In [ ]:
scene_texts = [
    "astronaut on crystal planet under galaxy sky",
    "crystal towers cracking with time loop energy",
    "astronaut standing still in time loop chaos",
    "planet stabilized with peaceful crystal towers"
]

In [ ]:
def clip_score(image, text):
    inputs = processor(
        text=[text],
        images=image,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    outputs = clip_model(**inputs)
    return outputs.logits_per_image.item()

In [ ]:
structured_scores = [
    clip_score(img, txt)
    for img, txt in zip(structured_scenes, scene_texts)
]

structured_clip = sum(structured_scores) / len(structured_scores)

In [ ]:
unstructured_scores = []

for text in scene_texts:
    patch_scores = [
        clip_score(patch, text)
        for patch in unstructured_scenes
    ]

    # PURE average (no max bias)
    unstructured_scores.append(sum(patch_scores)/len(patch_scores))

unstructured_clip = sum(unstructured_scores) / len(unstructured_scores)

In [ ]:
def compute_ssim(img1, img2):
    img1 = np.array(img1.resize((256,256))) / 255.0
    img2 = np.array(img2.resize((256,256))) / 255.0

    return ssim(img1, img2, channel_axis=2, data_range=1.0)

ssim_score = compute_ssim(img_unstructured, img_structured)

In [ ]:
def to_tensor(img):
    img = img.resize((256,256))
    img = np.array(img).astype(np.float32) / 255.0
    img = torch.tensor(img).permute(2,0,1).unsqueeze(0)
    return img * 2 - 1  # [-1,1]

lpips_score = lpips_model(
    to_tensor(img_unstructured),
    to_tensor(img_structured)
).item()

In [ ]:
def entity_consistency(images):
    scores = []

    for i in range(len(images)-1):
        score = lpips_model(
            to_tensor(images[i]),
            to_tensor(images[i+1])
        ).item()

        scores.append(score)

    return sum(scores) / len(scores)

entity_score = entity_consistency(structured_scenes)

In [ ]:
print("\n===== STORY 6 RESULTS =====")
print("CLIP Unstructured:", unstructured_clip)
print("CLIP Structured:", structured_clip)
print("SSIM:", ssim_score)
print("LPIPS:", lpips_score)
print("Entity Consistency:", entity_score)


===== STORY 6 RESULTS =====
CLIP Unstructured: 23.778572691811455
CLIP Structured: 23.609740257263184
SSIM: 0.036685822753422935
LPIPS: 0.6108106970787048
Entity Consistency: 0.6378286282221476


In [ ]:
#story 7

scene_texts = [
    "astronaut falling through wormhole into dark futuristic earth",
    "astronaut telling stories to people in dark city",
    "people looking at sky as light returns",
    "bright earth with spacecraft launching into space"
]

In [ ]:
from PIL import Image

img_unstructured = Image.open("/content/Unstructured_Story_7.png").convert("RGB")
img_structured = Image.open("/content/Structured_Story_7.png").convert("RGB")

In [ ]:
def split_2x2(image):
    w, h = image.size
    return [
        image.crop((0,0,w//2,h//2)),
        image.crop((w//2,0,w,h//2)),
        image.crop((0,h//2,w//2,h)),
        image.crop((w//2,h//2,w,h))
    ]

def split_3x3(image):
    w, h = image.size
    patches = []
    for i in range(3):
        for j in range(3):
            patches.append(image.crop((j*w//3, i*h//3, (j+1)*w//3, (i+1)*h//3)))
    return patches

structured_scenes = split_2x2(img_structured)
unstructured_scenes = split_3x3(img_unstructured)

In [ ]:
def clip_score(image, text):
    inputs = processor(
        text=[text],
        images=image,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    outputs = clip_model(**inputs)
    return outputs.logits_per_image.item()

In [ ]:
structured_scores = [
    clip_score(img, txt)
    for img, txt in zip(structured_scenes, scene_texts)
]

structured_clip = sum(structured_scores) / len(structured_scores)

In [ ]:
unstructured_scores = []

for text in scene_texts:
    patch_scores = [
        clip_score(patch, text)
        for patch in unstructured_scenes
    ]

    unstructured_scores.append(sum(patch_scores)/len(patch_scores))

unstructured_clip = sum(unstructured_scores) / len(unstructured_scores)

In [ ]:
import numpy as np
from skimage.metrics import structural_similarity as ssim

def compute_ssim(img1, img2):
    img1 = np.array(img1.resize((256,256))) / 255.0
    img2 = np.array(img2.resize((256,256))) / 255.0

    return ssim(img1, img2, channel_axis=2, data_range=1.0)

ssim_score = compute_ssim(img_unstructured, img_structured)

In [ ]:
def to_tensor(img):
    img = img.resize((256,256))
    img = np.array(img).astype(np.float32) / 255.0
    img = torch.tensor(img).permute(2,0,1).unsqueeze(0)
    return img * 2 - 1

lpips_score = lpips_model(
    to_tensor(img_unstructured),
    to_tensor(img_structured)
).item()

In [ ]:
def entity_consistency(images):
    scores = []

    for i in range(len(images)-1):
        score = lpips_model(
            to_tensor(images[i]),
            to_tensor(images[i+1])
        ).item()

        scores.append(score)

    return sum(scores) / len(scores)

entity_score = entity_consistency(structured_scenes)

In [ ]:
print("\n===== STORY 7 RESULTS =====")
print("CLIP Unstructured:", unstructured_clip)
print("CLIP Structured:", structured_clip)
print("SSIM:", ssim_score)
print("LPIPS:", lpips_score)
print("Entity Consistency:", entity_score)


===== STORY 7 RESULTS =====
CLIP Unstructured: 22.712351454628838
CLIP Structured: 25.091248989105225
SSIM: 0.1088941998708896
LPIPS: 0.4973202645778656
Entity Consistency: 0.5447474320729574
